In [16]:
# =============================================================================
# Exercise 3.45 — Symbolic Solution
# =============================================================================

import sympy as sp
from IPython.display import display, Math, Markdown

sp.init_printing(use_unicode=True)

z, n, k = sp.symbols('z n k', complex=True)


# =============================================================================
# Given transfer function
# =============================================================================

display(Markdown("## Given system"))

H = sp.factor((z**(-1) + sp.Rational(1, 2)*z**(-2)) / (1 - sp.Rational(3, 5)*z**(-1) + sp.Rational(2, 25)*z**(-2)))

display(Math(r"\mathcal{H}(z)=" + sp.latex(H)))

H_rational = sp.factor(sp.together(H))

display(Math(r"\mathcal{H}(z)=" + sp.latex(H_rational)))


# =============================================================================
# Poles of the system
# =============================================================================

display(Markdown("## Poles"))

D = 1 - sp.Rational(3, 5)*z**(-1) + sp.Rational(2, 25)*z**(-2)

poles = sp.solve(sp.Eq(sp.together(D), 0), z)

display(Math(r"D(z)=" + sp.latex(D)))

display(Math(r"\text{Factored denominator: }" + sp.latex(sp.factor(D))))

display(Math(r"\text{Poles: }" + ", ".join(sp.latex(p) for p in poles)))


# =============================================================================
# (a) Impulse response
# =============================================================================

display(Markdown("## (a) Impulse response"))

# H(z) = z^(-1) Y1(z)
# Therefore:
# Y1(z) = z H(z)

Y1 = sp.factor(sp.simplify(z * H))

display(Math(r"\mathcal{H}(z)=z^{-1}\mathcal{Y}_1(z)"))

display(Math(r"\mathcal{Y}_1(z)=" + sp.latex(Y1)))


# -----------------------------------------------------------------------------
# Residue calculation
#
# For the inverse Z-transform:
#
# x[n] = (1/2*pi*j) integral X(z) z^(n-1) dz
#
# Therefore the residue function must be:
#
# Y1(z) z^(n-1)
#
# NOT Y1(z) z^n
# -----------------------------------------------------------------------------

Fh = sp.factor(Y1 * z**(n - 1))

display(Math(r"\mathcal{Y}_1(z)z^{n-1}=" + sp.latex(Fh)))


# Calculate the residues at the poles

res_h = [sp.factor(sp.simplify(sp.residue(Fh, z, p))) for p in poles]

for p, r in zip(poles, res_h):
    display(Math(r"\operatorname{Res}_{z=" + sp.latex(p) + r"}=" + sp.latex(r)))


# Sum of residues

h1 = sp.factor(sp.simplify(sum(res_h)))

display(Math(r"\boxed{h_1[n]=" + sp.latex(h1) + r"}"))


# Since
#
# H(z) = z^(-1) Y1(z)
#
# the time-shift property gives:
#
# h[n] = h1[n-1]

h = sp.factor(sp.simplify(h1.subs(n, n - 1)))

display(Math(r"\boxed{h[n]=\left(" + sp.latex(h) + r"\right)u[n-1]}"))


# =============================================================================
# (b) Zero-state response for x[n] = u[n]
# =============================================================================

display(Markdown("## (b) Zero-state response"))

# x[n] = u[n]

display(Math(r"x[n]=u[n]"))


# -----------------------------------------------------------------------------
# Direct symbolic convolution
#
# y_ZS[n] = x[n] * h[n]
#
# Since
#
# h[n] = h1[n-1] u[n-1]
#
# and
#
# x[n] = u[n],
#
# we have
#
# y_ZS[n]
# = sum_{k=0}^{n} h[n-k]
# = sum_{k=0}^{n-1} h1[n-k-1]
# = sum_{m=0}^{n-1} h1[m]
# -----------------------------------------------------------------------------

k = sp.symbols('k', integer=True, nonnegative=True)

yzs_conv = sp.factor(sp.simplify(sp.summation(h1.subs(n, k), (k, 0, n - 1))))

display(Math(r"y_{ZS}[n]=x[n]*h[n]"))

display(Math(r"y_{ZS}[n]=\sum_{k=0}^{n}h[n-k]"))

display(Math(r"y_{ZS}[n]=\sum_{m=0}^{n-1}h_1[m]"))

display(Math(r"\boxed{y_{ZS}[n]=" + sp.latex(yzs_conv) + r"}"))


# -----------------------------------------------------------------------------
# Equivalent form of the convolution
# -----------------------------------------------------------------------------

yzs_conv_alt = sp.factor(sp.simplify(sp.summation(h1.subs(n, n - k - 1), (k, 0, n - 1))))

display(Math(r"y_{ZS,\mathrm{conv}}[n]=" + sp.latex(yzs_conv_alt)))


# -----------------------------------------------------------------------------
# Symbolic verification
# -----------------------------------------------------------------------------

convolution_check = sp.simplify(yzs_conv - yzs_conv_alt)

display(Math(r"\text{Convolution verification: }\quad " + sp.latex(convolution_check)))


# =============================================================================
# (c) Step response with initial conditions
# =============================================================================

display(Markdown("## (c) Step response with initial conditions"))


# -----------------------------------------------------------------------------
# Difference equation obtained directly from H(z)
# -----------------------------------------------------------------------------

display(Math(r"\mathcal{H}(z)=" + r"\frac{\mathcal{Y}(z)}{\mathcal{X}(z)}"))

display(Math(r"""
y[n]-\frac{3}{5}y[n-1]+\frac{2}{25}y[n-2]
=
x[n-1]+\frac{1}{2}x[n-2]
"""))


# -----------------------------------------------------------------------------
# Initial conditions
# -----------------------------------------------------------------------------

ym1 = sp.Integer(1)
ym2 = sp.Integer(2)

display(Math(r"y[-1]=" + sp.latex(ym1) + r",\qquad y[-2]=" + sp.latex(ym2)))


# -----------------------------------------------------------------------------
# Unilateral Z-transform of the homogeneous equation
# -----------------------------------------------------------------------------

Yp = sp.Symbol("Y^+(z)")

Yshift1 = z**(-1) * Yp + ym1

Yshift2 = z**(-2) * Yp + z**(-1) * ym1 + ym2

display(Math(r"\mathcal{Z}^{+}\{y[n-1]\}=" + sp.latex(Yshift1)))

display(Math(r"\mathcal{Z}^{+}\{y[n-2]\}=" + sp.latex(Yshift2)))


# Homogeneous equation

equation = sp.expand(Yp - sp.Rational(3, 5) * Yshift1 + sp.Rational(2, 25) * Yshift2)

display(Math(r"\text{Homogeneous equation: }" + sp.latex(equation) + r"=0"))


# Solve symbolically for Y^+(z)

Yzi = sp.factor(sp.solve(sp.Eq(equation, 0), Yp)[0])

display(Math(r"\boxed{\mathcal{Y}_{ZI}(z)=" + sp.latex(Yzi) + r"}"))


# -----------------------------------------------------------------------------
# Inverse Z-transform by residues
# -----------------------------------------------------------------------------

poles_zi = sp.solve(sp.Eq(sp.denom(sp.together(Yzi)), 0), z)

display(Math(r"\text{Poles: }" + ", ".join(sp.latex(p) for p in poles_zi)))


# Inverse Z-transform kernel

Fzi = sp.factor(Yzi * z**(n - 1))

display(Math(r"\mathcal{A}(z)=" + sp.latex(Fzi)))


# Residues

res_zi = [sp.factor(sp.simplify(sp.residue(Fzi, z, p))) for p in poles_zi]

for p, r in zip(poles_zi, res_zi):
    display(Math(r"\operatorname{Res}_{z=" + sp.latex(p) + r"}=" + sp.latex(r)))


# Zero-input response

yzi = sp.factor(sp.simplify(sum(res_zi)))

display(Math(r"\boxed{y_{ZI}[n]=" + sp.latex(yzi) + r"}"))


# =============================================================================
# Total response
# =============================================================================

display(Markdown("## Total response"))

ytotal = sp.factor(sp.simplify(yzs_conv + yzi))

display(Math(r"y[n]=y_{ZS}[n]+y_{ZI}[n]"))

display(Math(r"\boxed{y[n]=" + sp.latex(ytotal) + r"}"))

## Given system

<IPython.core.display.Math object>

<IPython.core.display.Math object>

## Poles

<IPython.core.display.Math object>

...

<IPython.core.display.Math object>

<IPython.core.display.Math object>

## Total response

<IPython.core.display.Math object>

<IPython.core.display.Math object>